# 17 — Predicting future sampled-BSADF state onsets

**Goal:** predict a new transition from a negative to a positive sampled-BSADF state within the next 30 or 60 daily observations. A current positive state is not eligible for a new-onset forecast. This is **not** independent proof of an economic/fundamental bubble.

The sampled-BSADF score and LPPLS features are observed only as of the decision date. Outcomes are censored unless a full future horizon is available. Train/validation/test splits are chronological and purge overlapping label horizons. Hyperparameters are fixed beforehand; the benchmark is separate from nested tuning in notebook 08b.

In [ ]:
from pathlib import Path
import sys
ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
import pandas as pd
from IPython.display import display
from run_statistical_benchmarks import run_experiment
from report_statistical_benchmarks import make_figures
FEATURES = ROOT/'08b_wyniki_tuning'/'01_cechy_dzienne.csv'
REFERENCE = ROOT/'13_wyniki_bubble'/'13_bubble_reference_daily.csv'
for path in [FEATURES, REFERENCE]:
    if not path.is_file():
        raise FileNotFoundError(path)


In [ ]:
from lppls_research.event_benchmarks import future_onset_labels
ref = pd.read_csv(REFERENCE,parse_dates=['date','reference_available_at'])
state = ref.bubble_state_gsadf.astype('Float64')
onsets = state.eq(1) & state.shift(1).eq(0)
print('Total historical positive-state starts:', int(onsets.fillna(False).sum()))
display(ref.loc[onsets.fillna(False), ['date','bsadf_score_last','bsadf_critical_last']])


In [ ]:
from event_level_validation import score_events
for horizon in (30,60):
    folder = ROOT/'17_wyniki_future_bubble'/f'onset_{horizon}d'
    result=run_experiment(FEATURES,REFERENCE,folder,
                          task='future_onset',horizon=horizon)
    make_figures(folder,f'Future sampled-BSADF onset, H={horizon} days',block=60)
    event_summary=score_events(result['predictions'],ref,horizon)
    event_summary.to_csv(folder/'event_level_summary.csv',index=False)
    print('Horizon:',horizon,'Eligible positives:',result['metadata']['overall_positive_dates'])
    display(result['summary'].sort_values('AP',ascending=False))
    display(event_summary.loc[event_summary.model.eq('XGBoost')])


## Interpretation

The positive-labeled **days** are not independent episodes. Evaluate event recall and alarm lead-time beside daily AP, and report small OOS episode counts. `BSADF-only` and the raw BSADF-gap rule are causal statistical comparators; neither should be benchmarked against the **same current-state label** as though it were an independent target. English-language vector figures and CSVs are written into `17_wyniki_future_bubble`.